# 06 Robust Proxy Judger

05 showed that a monotonic greedy soup is not enough: the proxy itself must be
stable.  This notebook evaluates the self-generated candidates from 02/03/04/05
on several mini validation splits, then trains a lightweight calibrator to map
multi-split proxy statistics to full-protocol quality.

The intended judge behavior is conservative but dynamic:

- trust a candidate only if it is good across splits, not just on one lucky split
- learn the relation between proxy evidence and full mAP from previous full evals
- choose a per-round global candidate without adding external teachers


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path('/app/Object_Detection')
PROJECT = ROOT / 'dynamic_quality_aware_classwise_aggregation' / 'moe_dqa_judger'
OUT = PROJECT / 'output' / '06_robust_proxy_judger'
OUT


PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/06_robust_proxy_judger')

In [2]:
import subprocess, sys

cmd = [
    sys.executable,
    str(PROJECT / 'scripts' / 'run_06_robust_proxy_judger.py'),
    '--workspace-root', str(OUT),
    '--max-round', '6',
    '--mini-splits', '3',
    '--mini-images', '384',
    '--max-candidates', '44',
    '--per-result-file-topk', '12',
    '--select-topk-per-round', '1',
    '--lcb-lambda', '0.75',
    '--pred-lcb-slack', '0.020',
    '--val-batch-size', '32',
    '--notify-discord',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_06_robust_proxy_judger.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/06_robust_proxy_judger --max-round 6 --mini-splits 3 --mini-images 384 --max-candidates 44 --per-result-file-topk 12 --select-topk-per-round 1 --lcb-lambda 0.75 --pred-lcb-slack 0.020 --val-batch-size 32 --notify-discord


{
  "manifest": {
    "created_utc": "2026-05-13T14:37:55.125664+00:00",
    "protocol": "dqa_softmox_robust_proxy_judger_v1",
    "method": "multi-split validation proxy with full-score calibrator",
    "source_workspace": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa/output/01_dqa_fedmox_yolo_full",
    "workspace": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/06_robust_proxy_judger",
    "max_round": 6,
    "mini_splits": 3,
    "mini_images": 384,
    "lcb_lambda": 0.75,
    "papers_used": [
      "FedLAW/Revisiting weighted aggregation: learned weights and shrinkage",
      "pFedLA/FedLAMA: layer-wise aggregation",
      "Model soups: validation-gated checkpoint mixing"
    ]
  },
  "selected": [
    {
      "label": "warmup_g0",
      "path": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa/output/01_dqa_fedmox_yolo_full/checkpoints/round000_latent_dqamox_warmup.pt",
      "source":

CompletedProcess(args=['/opt/venv/bin/python3', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_06_robust_proxy_judger.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/06_robust_proxy_judger', '--max-round', '6', '--mini-splits', '3', '--mini-images', '384', '--max-candidates', '44', '--per-result-file-topk', '12', '--select-topk-per-round', '1', '--lcb-lambda', '0.75', '--pred-lcb-slack', '0.020', '--val-batch-size', '32', '--notify-discord'], returncode=0)

In [3]:
summary = pd.read_csv(OUT / 'stats' / '06_robust_proxy_summary.csv')
display(summary[['round','label','source','role','mean_score','std_score','proxy_lcb_score','pred_full_score','known_full_score','judger_score']].head(20))

selected = pd.read_csv(OUT / 'stats' / '06_selected_policy_full.csv')
display(selected[['round','label','source','role','mean_score','std_score','proxy_lcb_score','pred_full_score','map50','map50_95','score','eval_scope']])

print((OUT / '06_robust_proxy_judger_report.md').read_text())


,round,label,source,role,mean_score,std_score,proxy_lcb_score,pred_full_score,known_full_score,judger_score
0,1,03_mix_judger_policy_r001_prior02,03_mix_judger_policy,learned_mix,0.628650,0.043054,0.596359,0.574550,0.57455,0.574550
1,2,03_mix_judger_policy_r002_prior07,03_mix_judger_policy,learned_mix,0.627933,0.041476,0.596826,0.574549,0.57455,0.574549
2,2,04_delta_expert_optimizer_r002_best00_sur00_01,04_delta_expert_optimizer,learned_mix,0.628517,0.042982,0.596280,0.574546,0.57455,0.574546
3,2,r002_g,source,g,0.628983,0.042551,0.597070,0.574545,NaN,0.574545
4,2,04_delta_expert_optimizer_r002_best01_sur00_00,04_delta_expert_optimizer,learned_mix,0.628567,0.042961,0.596346,0.574504,0.57450,0.574504
5,1,r001_s,source,s,0.628983,0.042551,0.597070,0.574501,NaN,0.574501
6,2,r002_a,source,a,0.627467,0.042004,0.595964,0.574494,NaN,0.574494
7,2,02_mix_weight_optimizer_expanded_r002_best00_p...,02_mix_weight_optimizer_expanded,learned_mix,0.627933,0.041476,0.596826,0.574466,0.57455,0.574466
8,2,02_mix_weight_optimizer_expanded_r002_best02_s...,02_mix_weight_optimizer_expanded,learned_mix,0.627967,0.041476,0.596860,0.574459,0.57450,0.574459
9,1,02_mix_weight_optimizer_expanded_r001_best02_s...,02_mix_weight_optimizer_expanded,learned_mix,0.629767,0.042827,0.597646,0.573950,0.57395,0.573950


,round,label,source,role,mean_score,std_score,proxy_lcb_score,pred_full_score,map50,map50_95,score,eval_scope
0,0,warmup_g0,source,warmup,0.621883,0.038900,0.592709,0.569235,0.458,0.255,0.56825,full_total
1,1,03_mix_judger_policy_r001_prior02,03_mix_judger_policy,learned_mix,0.628650,0.043054,0.596359,0.574550,0.462,0.260,0.57455,known_full
2,2,03_mix_judger_policy_r002_prior07,03_mix_judger_policy,learned_mix,0.627933,0.041476,0.596826,0.574549,0.462,0.260,0.57455,known_full
3,3,02_mix_weight_optimizer_expanded_r003_best02_s...,02_mix_weight_optimizer_expanded,learned_mix,0.619550,0.044111,0.586467,0.571000,0.459,0.258,0.57100,known_full
4,4,04_delta_expert_optimizer_r004_best01_rand000,04_delta_expert_optimizer,learned_mix,0.616500,0.039674,0.586744,0.567100,0.456,0.256,0.56710,known_full
5,5,r005_a,source,a,0.607083,0.039635,0.577357,0.560416,0.450,0.253,0.55965,full_total
6,6,r006_a,source,a,0.599600,0.036950,0.571888,0.559905,0.445,0.250,0.55285,full_total


# DQA-SoftMoX Robust Proxy Judger 06

- created_utc: 2026-05-13T14:38:11.446498+00:00
- candidate_count: 44
- mini_splits: 3
- mini_images: 384
- calibrator_train_count: 25
- calibrator_loo_mae: 0.00055

## Selected Policy

| round | label | source | role | mean proxy | std | LCB | pred full | full mAP50 | full mAP50:95 | full score |
|---:|---|---|---|---:|---:|---:|---:|---:|---:|---:|
| 0 | warmup_g0 | source | warmup | 0.6219 | 0.0389 | 0.5927 | 0.5692 | 0.458 | 0.255 | 0.5683 |
| 1 | 03_mix_judger_policy_r001_prior02 | 03_mix_judger_policy | learned_mix | 0.6287 | 0.0431 | 0.5964 | 0.5745 | 0.462 | 0.260 | 0.5746 |
| 2 | 03_mix_judger_policy_r002_prior07 | 03_mix_judger_policy | learned_mix | 0.6279 | 0.0415 | 0.5968 | 0.5745 | 0.462 | 0.260 | 0.5746 |
| 3 | 02_mix_weight_optimizer_expanded_r003_best02_sur00_01 | 02_mix_weight_optimizer_expanded | learned_mix | 0.6196 | 0.0441 | 0.5865 | 0.5710 | 0.459 | 0.258 | 0.5710 |
| 4 | 04_delta_expert_optimizer_r004_best01_rand000 | 04_del